# public_150.json タスク分類
このノートブックは `public_150.json` を読み込み、各タスクを **何をするタスクか**（conversion / extraction / generation）で分類し、集計と出力を行います。

- Python 3.12
- Jupyter Lab / CLI どちらでも実行可能（`python -m nbconvert --to notebook --execute ...` も可）


In [1]:
import json
import re
from pathlib import Path
import pandas as pd

SRC_PATH = Path('public_150.json')  # 必要ならパスを変更
tasks = json.loads(SRC_PATH.read_text(encoding='utf-8'))
print(type(tasks), len(tasks))


<class 'list'> 150


In [2]:
FORMAT_ALIASES = {
    'JSON': 'json',
    'YAML': 'yaml',
    'XML': 'xml',
    'CSV': 'csv',
    'TOML': 'toml',
    'Text': 'text',
}
STRUCTURED_FORMATS = {'json','yaml','xml','csv','toml'}

def parse_task_name(task_name: str):
    m = re.match(r'^\s*(Text|CSV|JSON|YAML|XML|TOML)\s+to\s+(CSV|JSON|YAML|XML|TOML)\s*$', task_name, re.IGNORECASE)
    if not m:
        return None, None
    src = m.group(1).title()
    dst = m.group(2).title()
    return FORMAT_ALIASES.get(src, src.lower()), FORMAT_ALIASES.get(dst, dst.lower())

def infer_kind(query: str, src_fmt: str, dst_fmt: str) -> str:
    q = (query or '').lower()
    if ('convert the following' in q) or ('transform this' in q) or ('please convert' in q):
        return 'conversion'
    if ('extract the following' in q) and ('attributes' in q):
        return 'extraction'
    if (src_fmt == 'text') and (dst_fmt in STRUCTURED_FORMATS):
        return 'generation'
    if (src_fmt in STRUCTURED_FORMATS) and (dst_fmt in STRUCTURED_FORMATS):
        return 'conversion'
    return 'other'

def infer_subtype(query: str, kind: str, dst_fmt: str) -> str:
    q = (query or '').lower()
    if kind == 'generation':
        if 'summarize' in q:
            return 'structured_summarization'
        return 'structured_generation'
    if kind == 'conversion':
        return f'{dst_fmt}_conversion'
    if kind == 'extraction':
        return f'{dst_fmt}_extraction'
    return ''


In [3]:
rows = []
for t in tasks:
    src_fmt, dst_fmt = parse_task_name(t.get('task_name',''))
    kind = infer_kind(t.get('query',''), src_fmt, dst_fmt)
    subtype = infer_subtype(t.get('query',''), kind, dst_fmt)
    rows.append({
        'task_id': t.get('task_id'),
        'task_name': t.get('task_name'),
        'source_format': src_fmt,
        'target_format': dst_fmt,
        'task_kind': kind,
        'task_subtype': subtype,
        'output_type': t.get('output_type'),
        'rendering': t.get('rendering', False),
    })

df = pd.DataFrame(rows)
df.head(10)


,task_id,task_name,source_format,target_format,task_kind,task_subtype,output_type,rendering
0,p_7b3394e21698627665533715,Text to JSON,text,json,generation,structured_summarization,JSON,False
1,p_bb594bd2d86606dbd1d1823d,Text to JSON,text,json,generation,structured_generation,JSON,False
2,p_bb8bcd0930ae9f9e4f0692ae,CSV to JSON,csv,json,conversion,json_conversion,JSON,False
3,p_b3fcb16b0778d50065908799,CSV to JSON,csv,json,conversion,json_conversion,JSON,False
4,p_f05e4e4665858d7bd1db0e71,CSV to JSON,csv,json,conversion,json_conversion,JSON,False
5,p_0cc0d1708fce08c15c260262,CSV to JSON,csv,json,conversion,json_conversion,JSON,False
6,p_067c0c165398faf5e1414ee8,CSV to JSON,csv,json,conversion,json_conversion,JSON,False
7,p_e5e42850758c5a2b27112acb,CSV to JSON,csv,json,conversion,json_conversion,JSON,False
8,p_7c9cbaeef4064a37002f4405,CSV to JSON,csv,json,conversion,json_conversion,JSON,False
9,p_ced5b041360b01bd91f38fd2,CSV to JSON,csv,json,conversion,json_conversion,JSON,False


In [4]:
print('task_kind counts:')
display(df['task_kind'].value_counts())

print('\n(task_name, task_kind) cross-tab:')
display(pd.crosstab(df['task_name'], df['task_kind']))


task_kind counts:


task_kind
conversion    110
generation     40
Name: count, dtype: int64


(task_name, task_kind) cross-tab:


task_kind,conversion,generation
task_name,,
CSV to JSON,14,0
CSV to XML,3,0
CSV to YAML,6,0
JSON to CSV,4,0
JSON to XML,6,0
JSON to YAML,14,0
TOML to JSON,11,0
TOML to YAML,8,0
Text to CSV,0,6


In [5]:
# 出力
OUT_DIR = Path('public_150_outputs')
OUT_DIR.mkdir(exist_ok=True)

df.to_csv(OUT_DIR/'public_150_task_classification.csv', index=False)
df.to_json(OUT_DIR/'public_150_task_classification.json', orient='records', force_ascii=False, indent=2)

# jsonl（行単位）
with (OUT_DIR/'public_150_task_classification.jsonl').open('w', encoding='utf-8') as f:
    for r in df.to_dict(orient='records'):
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('written:', OUT_DIR)


written: public_150_outputs
